# RetainIQ — 01b Train / Test Split (Phase 1.5)

**Purpose:** Produce the frozen train/test split that ALL modeling notebooks use.
Run this notebook exactly once.  The output files must not be regenerated after this point.

**Last updated:** 2026-05-01  
**Input:**  `data/processed/customers_master.parquet`  
**Outputs:** `data/processed/train.parquet`, `data/processed/test.parquet`

**Locked decisions (CLAUDE.md §6):**
- Stratified 80/20 split on `churned` — preserves class ratio in both splits.
- `RANDOM_SEED = 42` — reproducibility.
- Test set is FROZEN after this notebook runs.  No peek, no re-split.

In [1]:
import sys
from pathlib import Path

for _candidate in [Path.cwd(), Path.cwd().parent]:
    if (_candidate / "src" / "config.py").exists():
        sys.path.insert(0, str(_candidate))
        break

import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import (
    CUSTOMERS_MASTER, TRAIN_PATH, TEST_PATH,
    RANDOM_SEED, TEST_SIZE, TARGET,
)

print("Imports OK")
print(f"  RANDOM_SEED : {RANDOM_SEED}")
print(f"  TEST_SIZE   : {TEST_SIZE}")
print(f"  TARGET      : {TARGET}")

Imports OK
  RANDOM_SEED : 42
  TEST_SIZE   : 0.2
  TARGET      : churned


## Stratified Split

Stratification on `churned` ensures the ~10.11% churn rate is preserved in both
training and test sets.  Without stratification, random chance could skew the
minority class ratio in one split, inflating or deflating generalisation metrics.

80/20 is standard for this data size (375K rows).  The 20% test set (~75K rows)
is large enough for stable metric estimates at any subgroup we might slice.

In [2]:
master = pd.read_parquet(CUSTOMERS_MASTER)
print(f"Master shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"Overall churn rate: {master[TARGET].mean():.2%}")

train, test = train_test_split(
    master,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=master[TARGET],
    shuffle=True,
)

print(f"\nTrain: {train.shape[0]:,} rows ({train.shape[0]/len(master):.0%}) — "
      f"churn rate {train[TARGET].mean():.2%}")
print(f"Test:  {test.shape[0]:,} rows  ({test.shape[0]/len(master):.0%}) — "
      f"churn rate {test[TARGET].mean():.2%}")

Master shape: 375,537 rows × 38 cols
Overall churn rate: 10.11%

Train: 300,429 rows (80%) — churn rate 10.11%
Test:  75,108 rows  (20%) — churn rate 10.11%


## Save and Freeze

Saving both splits to `data/processed/`.  After this cell runs, the test set
must not be touched again until final evaluation in Phase 3.

The assert below confirms that train + test reconstruct the full master,
with no row duplication or data loss.

In [3]:
train.to_parquet(TRAIN_PATH, compression="snappy", index=False)
test.to_parquet(TEST_PATH,  compression="snappy", index=False)

print(f"Saved: {TRAIN_PATH}  ({TRAIN_PATH.stat().st_size/1e6:.1f} MB)")
print(f"Saved: {TEST_PATH}   ({TEST_PATH.stat().st_size/1e6:.1f} MB)")

# Sanity checks
assert len(train) + len(test) == len(master), "Row count mismatch — check split logic"
assert set(train["wallet_id"]).isdisjoint(set(test["wallet_id"])), \
    "wallet_id overlap between train and test — data leak!"

print("\nAll assertions passed.")
print("TEST SET IS NOW FROZEN. Do not re-run this notebook.")

Saved: C:\Users\User\Desktop\315 - retain IQ\retainiq\data\processed\train.parquet  (27.1 MB)
Saved: C:\Users\User\Desktop\315 - retain IQ\retainiq\data\processed\test.parquet   (7.7 MB)

All assertions passed.
TEST SET IS NOW FROZEN. Do not re-run this notebook.


---
> **FREEZE NOTICE — Phase 1.5 Complete**
>
> `data/processed/test.parquet` is the held-out evaluation set.
> It may not be used for:
> - Feature engineering decisions
> - Hyperparameter selection
> - SMOTE / scaling / encoding fits
> - Cross-validation folds
>
> It is used exactly once: final model evaluation in each Phase 2 notebook,
> after all training-side decisions are locked.  Any transformation applied
> to the test set must be fit on training data only and applied via
> `sklearn.pipeline.Pipeline`.